# Memory bank quality check

Verifies the two things `backend.memory_bank` needs to get right: (1) an added fact actually shows up naturally in a normal chat reply, and (2) `generate_reminiscence_prompt` produces a warm, accurate conversation opener from a stored fact, across languages, without inventing anything beyond what was stored. Outputs saved on run.

In [1]:
import os
import sys
from pathlib import Path

if not (Path.cwd() / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from backend import chat, memory_bank
from backend.db import get_profile_by_role

elder = get_profile_by_role("elder")
elder.id

2026-07-29 22:35:56.223 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


'f54dfa07-f642-41c4-a9b6-ee96066ff2e8'

## Fact influences a normal chat reply

In [2]:
fact = "Was a primary school teacher for 30 years and still loves reading storybooks aloud."
memory_bank.add_fact(elder.id, elder.id, fact)

reply = chat.send_message(elder.id, "I'm a bit bored this afternoon, nothing to do.")
print(reply)

2026-07-29 22:35:56.230 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


How about picking up a storybook to read aloud? You always loved that. 📖 Or maybe a little time with your rose bushes, if the weather's nice. 🌹

Would you like to tell me about one of your favorite books or teaching memories?


## Reminiscence prompt, across languages

Same underlying fact (rose bushes + granddaughter Mei, added earlier in manual testing), checked in three languages.

In [3]:
for lang in ["English", "Mandarin Chinese", "Malay"]:
    opener = memory_bank.generate_reminiscence_prompt(elder.id, lang)
    print(f"=== {lang} ===")
    print(opener)
    print("-" * 60)

=== English ===
Hi there! I heard you have quite the green thumb, especially when it comes to your rose bushes — I bet they're looking beautiful this time of year. And I heard your granddaughter Mei comes to visit you on weekends, too — that must be something you really look forward to. How have you been?
------------------------------------------------------------


=== Mandarin Chinese ===
奶奶，最近您的玫瑰花开得怎么样了呀？天气好的时候记得去园子里看看它们，别太累着自己哦。对了，这周末美美是不是又要来看您呀？想到您能和她一起在花园里待着，晒晒太阳、聊聊天，心里就觉得特别温暖呢。
------------------------------------------------------------


=== Malay ===
Selamat pagi! 😊

Teringat pula saya tentang cerita percutian keluarga ke Bali pada 2019 tu. Mesti banyak kenangan indah dan momen bahagia bersama keluarga tercinta ketika itu.

Boleh cerita sikit tak apa yang paling berkesan sepanjang percutian tu? Saya suka dengar cerita-cerita macam ni. 🌴
------------------------------------------------------------
